En este archivo tendremos diferentes tecnicas de preprocesado para poder evaluarlas y compararlas posteriormente con los resultados obtenidos y asi poder 
certificar cual de ellas es la que mejor resultados obtiene.

In [11]:
import pandas as pd
import re
import spacy

nlp = spacy.load("en_core_web_sm")

def preprocesamiento_basico(texto):
    texto = str(texto).lower()
    texto = re.sub(r'[^a-z\s]', ' ', texto)
    texto = re.sub(r'\s+', ' ', texto).strip()
    
    doc = nlp(texto)
    tokens_limpios = [token.text for token in doc if not token.is_stop]
    
    return " ".join(tokens_limpios)

def preprocesamiento_lematizacion(texto):
    texto = str(texto).lower()
    texto = re.sub(r'[^a-z\s]', ' ', texto)
    texto = re.sub(r'\s+', ' ', texto).strip()
    
    doc = nlp(texto)
    tokens_lema = [token.lemma_ for token in doc if not token.is_stop]
    
    return " ".join(tokens_lema)

def preprocesamiento_filtrado_pos(texto):
    texto = str(texto).lower()
    texto = re.sub(r'[^a-z\s]', ' ', texto)
    texto = re.sub(r'\s+', ' ', texto).strip()
    
    doc = nlp(texto)
    categorias_permitidas = ['NOUN', 'PROPN', 'ADJ']
    
    tokens_filtrados = [
        token.lemma_ for token in doc 
        if not token.is_stop and token.pos_ in categorias_permitidas
    ]
    
    return " ".join(tokens_filtrados)

Tambien realiazremos un preprocesamiento con la libreria de NLTK para poder comparar los resultados con la libreria de spaCy

In [12]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

stemmer = PorterStemmer()
stop_words_nltk = set(stopwords.words('english'))

def preprocesamiento_stemming_basico(texto):
    """
    Combinación A: LimpiezaRegex + StopWords + Stemming.
    La técnica más clásica y rápida en PLN. Corta las palabras por su sufijo
    (ej. 'cancerous' -> 'cancer', 'diagnosed' -> 'diagnos').
    """
    texto = str(texto).lower()
    texto = re.sub(r'[^a-z\s]', ' ', texto)
    texto = re.sub(r'\s+', ' ', texto).strip()
    
    # Tokenizamos separando por espacios
    tokens = texto.split()
    
    # Aplicamos filtro de stopwords y luego el Stemmer
    tokens_stemmed = [
        stemmer.stem(palabra) 
        for palabra in tokens 
        if palabra not in stop_words_nltk
    ]
    
    return " ".join(tokens_stemmed)

lemmatizer = WordNetLemmatizer()

def preprocesamiento_lematizacion_nltk(texto):
    texto = str(texto).lower()
    texto = re.sub(r'[^a-z\s]', ' ', texto)
    texto = re.sub(r'\s+', ' ', texto).strip()
    
    # 1. Tokenizamos usando NLTK
    tokens = nltk.word_tokenize(texto)
    
    # 2. Lematizamos y filtramos las stop words en el mismo paso
    tokens_lema = [
        lemmatizer.lemmatize(palabra) 
        for palabra in tokens 
        if palabra not in stop_words_nltk
    ]
    
    return " ".join(tokens_lema)

In [13]:
ruta_entrada = "../Datos/Original/tcga_simple_train.csv"
df = pd.read_csv(ruta_entrada)

df['text_basico'] = df['text'].apply(preprocesamiento_basico)
df['text_lema'] = df['text'].apply(preprocesamiento_lematizacion)
df['text_pos'] = df['text'].apply(preprocesamiento_filtrado_pos)
df['text_stem_nltk'] = df['text'].apply(preprocesamiento_stemming_basico)
df['text_lema_nltk'] = df['text'].apply(preprocesamiento_lematizacion_nltk)

ruta_salida = "../Datos/Preprocesados/tcga_preprocesado.csv"
df.to_csv(ruta_salida, index=False)